[Course material - Sustain.Brussels - "Avdanced AI workflows with LLM" - 20/04/2026 - 22/04/2026](https://github.com/Yannael/gen-ai-sustain-brussels-2604).

# Chat Templates & Instruct Models: Base vs Instruct

This notebook explains **what chat templates are**, why instruct-tuned models need them, and what happens when you skip them — using [Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B) (base) and [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) as a concrete example.

**What we'll cover:**
1. What is a chat template and why it exists
2. Downloading the base and instruct models
3. Inspecting the raw chat template
4. Applying the template to a conversation
5. Tokenizing and visualizing the result
6. Comparing base vs instruct on the same questions
7. What happens if you forget the template

## 1. Install dependencies

In [ ]:
!pip install transformers torch matplotlib --quiet

## 2. What is a chat template?

A **base model** (e.g. `Qwen2.5-0.5B`) is trained to predict the next token on raw text. It has no concept of "user" or "assistant" turns — it will just continue whatever text you give it.

An **instruct model** (e.g. `Qwen2.5-0.5B-Instruct`) is fine-tuned on conversations that were serialized in a specific format. At training time, every conversation looked something like:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
```

At inference time, you **must** reproduce this exact format — otherwise the model sees gibberish it was never trained on. The **chat template** is a template stored inside the tokenizer that does this serialization for you.

## 3. Download base and instruct models

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

BASE_MODEL    = "Qwen/Qwen2.5-0.5B"
INSTRUCT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading base tokenizer + model...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model     = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
base_model.eval()

print("Loading instruct tokenizer + model...")
instruct_tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)
instruct_model     = AutoModelForCausalLM.from_pretrained(INSTRUCT_MODEL, torch_dtype=torch.bfloat16)
instruct_model.eval()

print()
print(f"Base     — vocab size: {base_tokenizer.vocab_size:,}  params: {sum(p.numel() for p in base_model.parameters()):,}")
print(f"Instruct — vocab size: {instruct_tokenizer.vocab_size:,}  params: {sum(p.numel() for p in instruct_model.parameters()):,}")

Loading base tokenizer + model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Base     — vocab size: 151,643  params: 494,032,768
Instruct — vocab size: 151,643  params: 494,032,768


In [ ]:
base_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## 4. Inspect the raw chat template

The chat template is a Jinja2 string stored in the tokenizer's config. Let's print it out.

In [ ]:
print("=== BASE tokenizer chat template ===")
print(base_tokenizer.chat_template or "(none — base model has no chat template)")
print()
print("=" * 60)
print()
print("=== INSTRUCT tokenizer chat template ===")
print(instruct_tokenizer.chat_template)

=== BASE tokenizer chat template ===
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are a helpful assistant.<|im_end|

## 5. Apply the chat template

`tokenizer.apply_chat_template()` takes a list of `{"role": ..., "content": ...}` dicts and renders them into the format the model expects. Let's see the raw string it produces.

In [ ]:
conversation = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "What is the capital of France?"},
]

# tokenize=False → return the rendered string, not token IDs
rendered = instruct_tokenizer.apply_chat_template(
    conversation,
    tokenize=False,
    add_generation_prompt=True,   # appends the opening assistant turn
)

print("Rendered prompt (what actually gets fed to the model):")
print(repr(rendered))   # repr shows special characters clearly
print()
print("Human-readable version:")
print(rendered)

Rendered prompt (what actually gets fed to the model):
'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\n'

Human-readable version:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



## 6. Tokenize the prompt

Now let's tokenize the rendered prompt and show each token, its ID, and which part of the conversation it belongs to.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Encode to token IDs
input_ids = instruct_tokenizer.apply_chat_template(
    conversation,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    #enable_thinking=True
)



In [ ]:
input_ids

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,   3838,    374,    279,   6722,
            315,   9625,     30, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}

In [ ]:
token_ids   = input_ids['input_ids'][0].tolist()
token_strs  = [instruct_tokenizer.decode([t]) for t in token_ids]

print(f"Total tokens in prompt: {len(token_ids)}")
print()
print(f"{'#':>3}  {'ID':>6}  Token")
print("-" * 35)
for i, (tid, tstr) in enumerate(zip(token_ids, token_strs)):
    print(f"{i:>3}  {tid:>6}  {repr(tstr)}")

Total tokens in prompt: 26

  #      ID  Token
-----------------------------------
  0  151644  '<|im_start|>'
  1    8948  'system'
  2     198  '\n'
  3    2610  'You'
  4     525  ' are'
  5     264  ' a'
  6   10950  ' helpful'
  7   17847  ' assistant'
  8      13  '.'
  9  151645  '<|im_end|>'
 10     198  '\n'
 11  151644  '<|im_start|>'
 12     872  'user'
 13     198  '\n'
 14    3838  'What'
 15     374  ' is'
 16     279  ' the'
 17    6722  ' capital'
 18     315  ' of'
 19    9625  ' France'
 20      30  '?'
 21  151645  '<|im_end|>'
 22     198  '\n'
 23  151644  '<|im_start|>'
 24   77091  'assistant'
 25     198  '\n'


## 7. Generating a response with the instruct model

We pass the tokenized prompt to `model.generate()`. The model fills in the assistant turn.

In [ ]:
def instruct_chat(messages, max_new_tokens=1000, temperature=0.1):
    """Run a conversation through the instruct model."""
    ids = instruct_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        #enable_thinking=True
    )['input_ids'].to(instruct_model.device)
    prompt_len = ids.shape[1]

    with torch.no_grad():
        out = instruct_model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature,
            pad_token_id=instruct_tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_ids = out[0][prompt_len:]
    return instruct_tokenizer.decode(new_ids, skip_special_tokens=True), new_ids.tolist()


messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "What is 2+2?"},
]

response, gen_ids = instruct_chat(messages)
print("Instruct model response:")
print(response)
print()
print(f"Generated {len(gen_ids)} tokens: {gen_ids}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Instruct model response:
The answer to 2 + 2 is 4.

This is a simple arithmetic operation that involves adding two numbers together. In this case, we have:

2 + 2 = 4

So the result of 2 + 2 is 4.

Generated 54 tokens: [785, 4226, 311, 220, 17, 488, 220, 17, 374, 220, 19, 382, 1986, 374, 264, 4285, 34784, 5666, 429, 17601, 7842, 1378, 5109, 3786, 13, 758, 419, 1142, 11, 582, 614, 1447, 17, 488, 220, 17, 284, 220, 19, 271, 4416, 279, 1102, 315, 220, 17, 488, 220, 17, 374, 220, 19, 13, 151645]


## 8. Base model — no template, just text continuation

The base model knows nothing about `<|im_start|>` formatting. We give it a plain text prompt and it continues the text.

In [ ]:
def base_complete(prompt, max_new_tokens=80, temperature=0.1):
    """Run plain text completion through the base model."""
    ids = base_tokenizer.encode(prompt, return_tensors="pt")
    prompt_len = ids.shape[1]

    with torch.no_grad():
        out = base_model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature,
            pad_token_id=base_tokenizer.eos_token_id,
        )

    new_ids = out[0][prompt_len:]
    return base_tokenizer.decode(new_ids, skip_special_tokens=True), new_ids.tolist()


prompt = "What is 2+2?"
response, gen_ids = base_complete(prompt)
print(f"Prompt : {prompt}")
print(f"Base continuation:")
print(response)
print()
print(f"Generated {len(gen_ids)} tokens: {gen_ids}")

Prompt : What is 2+2?
Base continuation:
 - Answers\nMath and Arithmetic\nWhat is 2+2?\nWiki User\n∙ 2010-04-28 17:48:06\nSee Answer\nBest Answer\nCopy\n2+2=4\nWiki User\n∙ 2010-04-28 17:48:06\nThis

Generated 80 tokens: [481, 37243, 1699, 8815, 323, 92984, 1699, 3838, 374, 220, 17, 10, 17, 32620, 77, 53896, 2657, 1699, 144498, 220, 17, 15, 16, 15, 12, 15, 19, 12, 17, 23, 220, 16, 22, 25, 19, 23, 25, 15, 21, 1699, 9830, 21806, 1699, 14470, 21806, 1699, 12106, 1699, 17, 10, 17, 28, 19, 1699, 53896, 2657, 1699, 144498, 220, 17, 15, 16, 15, 12, 15, 19, 12, 17, 23, 220, 16, 22, 25, 19, 23, 25, 15, 21, 1699, 1986]


## 9. Side-by-side comparison on several questions

Now let's run both models on the same questions and compare their outputs.

In [ ]:
questions = [
    "What is your name?",
    "What is the capital of France?",
    "What is 2 + 2?",
    "Explain gravity in one sentence.",
]

SYSTEM = "You are a helpful assistant."
SEP = "-" * 70

for q in questions:
    print(SEP)
    print(f"Question: {q}")
    print()

    # Base model — plain completion
    base_resp, base_ids = base_complete(q, max_new_tokens=60)
    print(f"[BASE]     {repr(base_resp[:200])}")
    print(f"           tokens generated: {len(base_ids)}")
    print()

    # Instruct model — with chat template
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    inst_resp, inst_ids = instruct_chat(msgs, max_new_tokens=60)
    print(f"[INSTRUCT] {repr(inst_resp[:200])}")
    print(f"           tokens generated: {len(inst_ids)}")
    print()

print(SEP)

----------------------------------------------------------------------
Question: What is your name?

[BASE]     ' I am a computer program designed to assist with various tasks. My name is "AI Assistant".'
           tokens generated: 20

[INSTRUCT] 'I am Qwen, an AI language model created by Alibaba Cloud.'
           tokens generated: 15

----------------------------------------------------------------------
Question: What is the capital of France?

[BASE]     ' The capital of France is Paris.'
           tokens generated: 8

[INSTRUCT] 'The capital of France is Paris.'
           tokens generated: 8

----------------------------------------------------------------------
Question: What is 2 + 2?

[BASE]     ' - Answers\\nMath and Arithmetic\\nWhat is 2 + 2?\\nWiki User\\n∙ 2011-04-26 17:55:26\\nSee Answer\\nBest Answer\\nCopy\\n2 + 2 = 4\\nWiki User\\n'
           tokens generated: 60

[INSTRUCT] 'The answer to 2 + 2 is 4.'
           tokens generated: 13

----------------------------

## 10. What happens if you feed the instruct model a raw prompt?

This is the most common mistake. Let's see what the instruct model does when it receives a plain question **without** the chat template.

In [ ]:
question = "What is the capital of France?"

# --- WRONG: no template ---
ids_no_template = instruct_tokenizer.encode(question, return_tensors="pt")
with torch.no_grad():
    out_no_template = instruct_model.generate(
        ids_no_template,
        max_new_tokens=60,
        pad_token_id=instruct_tokenizer.eos_token_id,
    )
no_template_resp = instruct_tokenizer.decode(
    out_no_template[0][ids_no_template.shape[1]:], skip_special_tokens=True
)

# --- RIGHT: with template ---
msgs = [{"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": question}]
correct_resp, _ = instruct_chat(msgs, max_new_tokens=60)

print(f"Question: {question}")
print()
print(f"[INSTRUCT — NO TEMPLATE]  {repr(no_template_resp[:300])}")
print()
print(f"[INSTRUCT — WITH TEMPLATE] {repr(correct_resp[:300])}")

Question: What is the capital of France?

[INSTRUCT — NO TEMPLATE]  ' The capital of France is Paris. \n\nTo verify this, let\'s break down the information:\n\n1. "The capital" refers to the largest city or administrative center in a country.\n2. Paris is the capital of France.\n\nTherefore, the answer to the question "What is the capital of France'

[INSTRUCT — WITH TEMPLATE] 'The capital of France is Paris.'


## Summary

| | Base model | Instruct model |
|---|---|---|
| **Training data** | Raw text | Conversations in a fixed format |
| **Input** | Plain text prompt | Rendered chat template |
| **Output** | Text continuation | Answers to the user's question |
| **Chat template** | Not needed | **Required** — model breaks without it |

**Key takeaway:** the chat template is a specific string format the instruct model was trained to expect. `tokenizer.apply_chat_template()` is the canonical way to produce that string.